# M2.S4 - Introduction to GPU and Accelerator Computing
## Interactive HPC notebook

This notebook accompanies **M2.S4 - Introduction to GPU and Accelerator Computing**.

The goal is to connect the slides to **real code and real measurements on the SciTech GPU**. For every experiment, first read the relevant C/CUDA/OpenACC code, predict the result, then run and explain it.

### What you will do

1. compare a tiny and a large workload on a real CPU and GPU;
2. see how CUDA block and thread IDs map to actual work;
3. measure the cost of copying data to and from the GPU;
4. run the same vector-add idea with explicit CUDA;
5. run an OpenACC version on the same NVIDIA GPU;
6. connect the programming model to the Slurm resource workflow.

### Classroom method

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

### One GPU allocation, several experiments

To avoid requesting a new GPU for every small activity, this notebook submits **one short Slurm GPU job**. That job runs all the real accelerator examples. Later sections reveal and discuss one part of the output at a time.

> **Concepts can be explained with text. Performance claims should be measured.**

> **Notebook build: M2S4-2026-09-22-v8**
>
> This version uses one real SciTech GPU mini-lab and shows the **small piece of C/CUDA/OpenACC code that matters before each experiment**.
>
> Setup and output-parsing code is kept in the background so the focus stays on GPU programming and performance.

# 0 - Check the environment

### Predict

Before running:

- Is this Jupyter kernel itself a GPU allocation?
- Which scheduler controls access to the shared GPU?
- What is the difference between seeing NVIDIA software and actually owning a GPU?

In [ ]:
import os
import platform
import re
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

print("Host:", platform.node())
print("User:", os.environ.get("USER", "unknown"))
print("Current Slurm job:", os.environ.get("SLURM_JOB_ID", "not set"))
print("Current Slurm partition:", os.environ.get("SLURM_JOB_PARTITION", "not set"))
print("sbatch:", shutil.which("sbatch"))
print("sinfo:", shutil.which("sinfo"))
print("nvidia-smi visible in PATH:", shutil.which("nvidia-smi"))

### Explain

Seeing `nvidia-smi`, CUDA libraries or NVIDIA tools on a system does **not** mean your current process owns a GPU.

On this cluster:

```text
Jupyter session
      |
      | sbatch requests a GPU
      v
Slurm GPU partition
      |
      v
GPU compute node
```

Slurm controls access to the accelerator.

# 1 - Prepare the real examples

The real C/CUDA/OpenACC programs used in this notebook live in:

```text
session_demos/09_gpu/
```

The setup cell below simply copies those files into the notebook working directory, or downloads them from GitHub if the repository is not already cloned.

> **This is notebook plumbing, not the GPU lesson.** You can run the cell without studying its Python implementation.

Full examples:
https://github.com/OscarDiez/hpc_course/tree/main/session_demos/09_gpu

In [ ]:
DEMO_NAMES = [
    "quick_compare.cu",
    "thread_mapping.cu",
    "data_movement.cu",
    "vector_add.cu",
    "openacc_vector_add.c",
    "gpu_lab.sbatch",
]

RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "OscarDiez/hpc_course/main/session_demos/09_gpu/"
)

LOCAL_DEMO = Path.home() / "hpc_course" / "session_demos" / "09_gpu"

for name in DEMO_NAMES:
    target = Path(name)

    if (LOCAL_DEMO / name).exists():
        shutil.copy2(LOCAL_DEMO / name, target)
        source = "local ~/hpc_course"
    else:
        urllib.request.urlretrieve(RAW_BASE + name, target)
        source = "GitHub"

    print(f"{name:<26} <- {source}")

print("\nExamples ready.")

# 2 - Request one real GPU

We will use **one short Slurm GPU job** for all five experiments.

### What resource are we asking for?

The relevant part of the batch script is:

```bash
#SBATCH --partition=gpu
#SBATCH --gpus=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=4G
#SBATCH --time=00:05:00
```

Inside the allocated compute node we load the compiler stack that we have validated on SciTech:

```bash
module purge
module load nvhpc/25.7
```

Then the job compiles and runs the CUDA and OpenACC programs.

### The workflow

```text
JupyterHub
    |
    | sbatch gpu_lab.sbatch
    v
Slurm
    |
    | allocates 1 GPU
    v
GPU compute node
    |
    | nvcc / nvc
    v
CUDA + OpenACC programs
```

### Predict

Before submitting:

1. Which GPU model do you expect?
2. Will a tiny 1,024-element vector addition favor the CPU or GPU end-to-end?
3. Will keeping data on the GPU help when the same operation is repeated many times?

In [ ]:
def clean_slurm_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit_slurm(script):
    p = subprocess.run(
        ["sbatch", "--parsable", script],
        capture_output=True,
        text=True,
        env=clean_slurm_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr.strip() or "sbatch failed")
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def wait_for_job(job_id, timeout=300, poll=3):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True,
            text=True
        )
        state = p.stdout.strip()
        if not state:
            print("Job", job_id, "has left the queue.")
            return True
        print("Job", job_id, "state:", state)
        time.sleep(poll)
    print("Timed out. The job may still be queued or running.")
    return False

def read_job_output(job_id):
    path = Path(f"m2s4_lab-{job_id}.out")
    if not path.exists():
        print("Output file not found yet:", path)
        return ""
    return path.read_text(errors="replace")

def show_section(text, number):
    start = f"===== EXPERIMENT {number}:"
    end = f"===== END EXPERIMENT {number} ====="

    i = text.find(start)
    j = text.find(end)

    if i < 0 or j < 0:
        print("Experiment section not found.")
        return

    print(text[i:j + len(end)])

print("Slurm helpers ready.")

In [ ]:
if shutil.which("sinfo"):
    print("--- GPU partition ---")
    subprocess.run(
        ["sinfo", "-p", "gpu", "-o", "%P %a %l %D %c %G"],
        check=False
    )

M2S4_JOB_ID = submit_slurm("gpu_lab.sbatch")

if wait_for_job(M2S4_JOB_ID, timeout=300):
    time.sleep(1)
    M2S4_LAB_OUTPUT = read_job_output(M2S4_JOB_ID)
    print("\nGPU mini-lab completed.")
else:
    M2S4_LAB_OUTPUT = ""

# 3 - Real example 1: when does the GPU win?
### Same operation, CPU vs GPU

We perform the **same vector addition** on the CPU and GPU.

Full source: `session_demos/09_gpu/quick_compare.cu`

## What are we running?

### CPU version

```cpp
for (int r = 0; r < reps; ++r)
    for (int i = 0; i < n; ++i)
        c[i] = a[i] + b[i];
```

One CPU thread repeatedly walks through the array.

### GPU version

```cpp
__global__ void add(const float *a,
                    const float *b,
                    float *c,
                    int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
        c[i] = a[i] + b[i];
}

for (int r = 0; r < reps; ++r)
    add<<<blocks, threads>>>(d_a, d_b, d_c, n);
```

Many GPU threads execute the same kernel on different elements.

## Two measured cases

- **tiny**: 1,024 elements, one operation;
- **large + reuse**: 5,000,000 elements, repeated 20 times.

For the GPU we measure:

- **GPU kernel time**: the arrays are already on the GPU;
- **GPU total time**: copies + kernels + result copy.

### Predict

1. Which case should favor the CPU?
2. Which case should favor the GPU?
3. Which GPU timing is the fair end-to-end comparison with the CPU?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 1)

In [ ]:
def parse_case(text, label):
    pattern = (
        rf"^{re.escape(label)}\s+(\d+)\s+(\d+)\s+"
        rf"([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+(PASS|FAIL)$"
    )
    m = re.search(pattern, text, re.MULTILINE)
    if not m:
        return None
    return {
        "N": int(m.group(1)),
        "reps": int(m.group(2)),
        "cpu_ms": float(m.group(3)),
        "kernel_ms": float(m.group(4)),
        "gpu_total_ms": float(m.group(5)),
        "check": m.group(6),
    }

tiny = parse_case(M2S4_LAB_OUTPUT, "tiny")
large = parse_case(M2S4_LAB_OUTPUT, "large_reuse")

for name, result in [("Tiny workload", tiny), ("Large + reuse", large)]:
    if not result:
        continue

    cpu = result["cpu_ms"]
    gpu = result["gpu_total_ms"]

    if gpu < cpu:
        print(
            f"{name}: GPU is {cpu/gpu:.1f}x faster end-to-end "
            f"(CPU {cpu:.4f} ms vs GPU {gpu:.4f} ms)"
        )
    else:
        print(
            f"{name}: CPU is {gpu/cpu:.1f}x faster end-to-end "
            f"(CPU {cpu:.4f} ms vs GPU {gpu:.4f} ms)"
        )

### Explain

The fair application comparison is:

```text
CPU time  vs  GPU total time
```

The kernel-only number answers a different question:

> How fast is the accelerator computation **after the data is already on the GPU**?

For a tiny workload, GPU launch and transfer overhead can be much larger than the useful computation.

For a large regular workload with reuse, the GPU has enough independent work to exploit its throughput.

> **Do not ask "Is the GPU faster?" Ask "Is the complete workload large and regular enough to justify using the GPU?"**

# 4 - Real example 2: CUDA blocks and threads

The slides introduce:

```c
int i = blockIdx.x * blockDim.x + threadIdx.x;
```

Now a real CUDA kernel records its own block and thread IDs.

Full source: `session_demos/09_gpu/thread_mapping.cu`

## What are we running?

```cpp
__global__ void record_mapping(int *block_id,
                               int *thread_id)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    block_id[i]  = blockIdx.x;
    thread_id[i] = threadIdx.x;
}
```

We launch:

```text
20 useful elements
8 threads per block
3 blocks
24 launched threads
```

### Predict

1. What global index is **block 1, thread 3**?
2. How many of the 24 launched threads have no useful element?
3. Why does normal vector code need `if (i < n)`?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 2)

### Explain

For one-dimensional CUDA indexing:

```text
global index = block number x threads per block + thread number
```

So:

```text
block 1, thread 3
= 1 x 8 + 3
= global index 11
```

The last block is only partly useful. CUDA launches whole blocks, so extra threads are normal.

For a vector of length `n`, the safety check is:

```c
if (i < n)
    c[i] = a[i] + b[i];
```

Without it, the extra threads could access memory beyond the end of the array.

# 5 - Real example 3: data movement can dominate

We now run the same 2,000,000-element vector operation 20 times in two different ways.

Full source: `session_demos/09_gpu/data_movement.cu`

## Strategy A - copy every iteration

```cpp
for (int r = 0; r < 20; ++r) {

    cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);

    add<<<blocks, threads>>>(d_a, d_b, d_c, n);

    cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);
}
```

Every operation pays the CPU-to-GPU and GPU-to-CPU transfer cost.

## Strategy B - keep the arrays on the GPU

```cpp
cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);

for (int r = 0; r < 20; ++r)
    add<<<blocks, threads>>>(d_a, d_b, d_c, n);

cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);
```

The mathematical work is the same. We have only moved the transfers **outside the loop**.

### Predict

Which version should be faster, and by how much do you think?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 3)

In [ ]:
copy_m = re.search(r"copy_every_iteration_ms=([0-9.]+)", M2S4_LAB_OUTPUT)
resident_m = re.search(r"keep_data_resident_ms=([0-9.]+)", M2S4_LAB_OUTPUT)

if copy_m and resident_m:
    copy_ms = float(copy_m.group(1))
    resident_ms = float(resident_m.group(1))

    print(f"Copy every iteration : {copy_ms:.3f} ms")
    print(f"Keep data resident   : {resident_ms:.3f} ms")
    print(f"Keeping data resident is {copy_ms/resident_ms:.2f}x faster in this run.")
else:
    print("Timing values not found.")

### Explain

The measured result demonstrates:

```text
GPU application time
=
copy in + compute + copy out
```

A fast GPU kernel does not guarantee a fast application.

Notice what changed in the code:

```text
BEFORE: copy -> kernel -> copy, 20 times
AFTER : copy -> 20 kernels -> copy
```

The calculation itself did not become faster. We simply stopped moving the same data unnecessarily.

> **Move data less often. Do more useful work while it is on the GPU.**

# 6 - Real example 4: minimal CUDA vector addition

This is the smallest complete CUDA example in the lab.

Full source: `session_demos/09_gpu/vector_add.cu`

## What are we running?

### 1. Allocate device memory

```cpp
cudaMalloc(&d_a, bytes);
cudaMalloc(&d_b, bytes);
cudaMalloc(&d_c, bytes);
```

### 2. Copy inputs from CPU memory to GPU memory

```cpp
cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice);
cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice);
```

### 3. Run the kernel

```cpp
__global__ void vector_add(const float *a,
                           const float *b,
                           float *c,
                           int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
        c[i] = a[i] + b[i];
}

int threads = 256;
int blocks = (n + threads - 1) / threads;

vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
```

### 4. Copy the result back

```cpp
cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost);
```

For this experiment:

```text
N = 1,024
threads per block = 256
blocks = 4
```

### Predict

How many useful vector elements does each GPU thread handle?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 4)

### Explain

Each useful CUDA thread handles **one vector element**.

The important CUDA sequence is:

```text
allocate GPU memory
        |
copy CPU -> GPU
        |
launch many GPU threads
        |
copy GPU -> CPU
```

CUDA gives explicit control over all four stages.

That control is useful, but it also creates more code and more responsibility for the programmer.

# 7 - Real example 5: the same idea with OpenACC

Now we express essentially the same vector-style accelerator computation with OpenACC.

Full source: `session_demos/09_gpu/openacc_vector_add.c`

## What are we running?

```c
#pragma acc data copyin(a[0:n], b[0:n]) copyout(c[0:n])
{
    #pragma acc parallel loop
    for (int i = 0; i < n; ++i)
        c[i] = a[i] + b[i];
}
```

The directives tell the compiler:

- copy `a` and `b` to the accelerator;
- create accelerator code for the loop;
- copy `c` back when the data region ends.

## Compare the programming models

### CUDA

```cpp
cudaMalloc(...);
cudaMemcpy(...);

vector_add<<<blocks, threads>>>(...);

cudaMemcpy(...);
```

### OpenACC

```c
#pragma acc data ...
#pragma acc parallel loop
for (...)
    ...
```

### Predict

1. Which approach gives more low-level control?
2. Which requires fewer source changes?
3. Does fewer lines of code automatically mean faster execution?

In [ ]:
show_section(M2S4_LAB_OUTPUT, 5)

### Explain

- **CUDA** gives explicit control over device memory, copies, grid/block geometry and kernel launches.
- **OpenACC** asks the compiler to create the accelerator implementation from directives.
- Both examples in this notebook execute on the real NVIDIA GPU.
- Fewer source changes do **not** guarantee better performance.
- More low-level control does **not** guarantee better performance either.

The right question is:

> **What is the highest-level programming model that gives this application the control and performance it needs?**

# 8 - Two ways to reach the same SciTech GPU

## From this JupyterHub notebook

Jupyter is already inside a Slurm-managed allocation, so this notebook submits:

```bash
sbatch gpu_lab.sbatch
```

The workflow is:

```text
JupyterHub -> sbatch -> GPU partition -> GPU compute node
```

## From the normal SSH login node

For a short interactive demonstration:

```bash
srun -p gpu --gpus=1 --cpus-per-task=2 --mem=4G --time=00:10:00 --pty bash -l
```

Then:

```bash
hostname
nvidia-smi

module purge
module load nvhpc/25.7
```

The CUDA/OpenACC programming model is the same. Only the **resource-allocation workflow** changes.

> Do not start the interactive `srun --pty` route from inside the existing Jupyter Slurm allocation.

# 9 - Which programming approach would you choose?

Choose a sensible first approach.

### A
A 10,000-line scientific C application has one loop that consumes 70% of runtime.

### B
A new NVIDIA-specific algorithm needs explicit control of threads, memory and execution.

### C
The hot operation is standard dense matrix multiplication.

<details>
<summary><strong>Show suggested answer</strong></summary>

- **A - OpenACC** is a plausible first approach because it can accelerate existing loop-based code with relatively small source changes.
- **B - CUDA** fits when NVIDIA-specific low-level control is important.
- **C - optimized GPU library** should usually be tried first when a high-quality implementation already exists.

</details>

# 10 - Challenge: diagnose a GPU result

A team reports:

```text
CPU application       = 200 ms
GPU kernel            =  20 ms
complete GPU program  = 150 ms
```

Answer:

1. Is the GPU kernel 10x faster than the CPU computation?
2. Is the application 10x faster?
3. What is probably consuming the missing time?
4. What could help if the same data is used by 50 GPU kernels?

<details>
<summary><strong>Show suggested solution</strong></summary>

1. Yes. At kernel level, 200 / 20 = 10x.
2. No. End-to-end speedup is only 200 / 150 = 1.33x.
3. Investigate transfers, allocation, synchronization and launch overhead.
4. Keep the data resident on the GPU and perform many kernels before copying the final result back.

</details>

# What did we learn?

1. **A GPU is not automatically faster.** Small work can favor the CPU.
2. **GPUs are good at throughput.** Large, regular, independent work is a natural fit.
3. **CUDA maps work through grids, blocks and threads.**
4. **Extra threads are normal.** Boundary checks keep them safe.
5. **Data movement matters.** Repeated copies can dominate runtime.
6. **Keeping data resident can change the result dramatically.**
7. **CUDA and OpenACC can run the same idea at different abstraction levels.**
8. **Measure end-to-end performance, not only kernel time.**
9. **Use Slurm when you actually need the shared GPU.**

### Source examples

The programs used by this notebook are also available here:

```text
https://github.com/OscarDiez/hpc_course/tree/main/session_demos/09_gpu
```

The next session continues the same HPC question:

> **Where is the real bottleneck, and what programming model matches the architecture?**